# Nokia Patent Portfolio Analysis — SQL Query Notebook

This notebook consolidates the repository’s BigQuery Standard SQL queries, one query per code cell, with a short note before each query.

**Execution status:** archival/reproducibility notebook. The queries require access to Google BigQuery, `patents-public-data.patents.publications`, and the temporary `nokia_analytics` sandbox tables where noted. They were originally run in the BigQuery console; the saved CSV outputs in `data/` are fingerprinted in `data/source_manifest.csv`. Do not run cells here without the required BigQuery project, credentials, and sandbox tables.

**Terminology:** SQL column aliases such as `owner` refer to applicant/assignee names at filing/publication, not verified current ownership. Grant evidence is not verified current legal status.


## Query files

- `sql/01_owner_names.sql`
- `sql/02_lineage_names.sql`
- `sql/03_filings_by_country.sql`
- `sql/04_check_us_kind_codes.sql`
- `sql/05_owner_names_all_years.sql`
- `sql/06_create_portfolio_30y.sql`
- `sql/07_check_portfolio_30y.sql`
- `sql/08_recount_us_30y.sql`
- `sql/09_family_status.sql`
- `sql/10_in_term_by_office.sql`
- `sql/11_new_families_by_year.sql`
- `sql/12_office_filings_by_family.sql`
- `sql/13_spot_check_sample.sql`
- `sql/14_mean_median_patent_measures.sql`
- `sql/15_rivals_30y.sql`
- `sql/16_codebook_sample.sql`
- `sql/17_codebook_sample_h04l49.sql`
- `sql/18_validation_sample_metadata.sql`


## `01_owner_names.sql`

Superseded by 05_owner_names_all_years.sql: the kind code filter below misses US patents granted before 2001.


In [ ]:
-- Superseded by 05_owner_names_all_years.sql: the kind code filter below misses US patents granted before 2001.
-- Every owner name attached to granted US patents that mentions Nokia or Alcatel.
SELECT
  a.name AS owner,
  COUNT(DISTINCT p.publication_number) AS granted_patents
FROM `patents-public-data.patents.publications` AS p,
  UNNEST(p.assignee_harmonized) AS a
WHERE p.country_code = 'US'
  AND p.grant_date > 0
  AND p.kind_code LIKE 'B%'
  AND (UPPER(a.name) LIKE '%NOKIA%' OR UPPER(a.name) LIKE '%ALCATEL%')
GROUP BY owner
ORDER BY granted_patents DESC;


## `02_lineage_names.sql`

Superseded by 05_owner_names_all_years.sql: the kind code filter below misses US patents granted before 2001.


In [ ]:
-- Superseded by 05_owner_names_all_years.sql: the kind code filter below misses US patents granted before 2001.
-- Owner names for companies absorbed through deals, left out of the first query.
-- \b matches whole words, so TRANSLUCENT is excluded. Alcatel names are already in query 01.
SELECT
  a.name AS owner,
  COUNT(DISTINCT p.publication_number) AS granted_patents
FROM `patents-public-data.patents.publications` AS p,
  UNNEST(p.assignee_harmonized) AS a
WHERE p.country_code = 'US'
  AND p.grant_date > 0
  AND p.kind_code LIKE 'B%'
  AND REGEXP_CONTAINS(UPPER(a.name), r'\b(LUCENT|BELL LABS|INFINERA)\b')
  AND NOT REGEXP_CONTAINS(UPPER(a.name), r'ALCATEL')
GROUP BY owner
ORDER BY granted_patents DESC;


## `03_filings_by_country.sql`

Where the Nokia family's patent applications were filed.


In [ ]:
-- Where the Nokia family's patent applications were filed.
-- EP is the European Patent Office, not the EU. Chinese-language names are included for CN records.
-- grant_date may be incomplete outside the US, so check granted_applications before trusting it.
SELECT
  p.country_code,
  COUNT(DISTINCT p.application_number) AS applications,
  COUNT(DISTINCT IF(p.grant_date > 0, p.application_number, NULL)) AS granted_applications,
  COUNT(DISTINCT p.family_id) AS patent_families
FROM `patents-public-data.patents.publications` AS p,
  UNNEST(p.assignee_harmonized) AS a
WHERE REGEXP_CONTAINS(
    UPPER(a.name),
    r'\bNOKIA\b|\bALCATEL\b|\bLUCENT TECHNOLOGIES\b|\bINFINERA\b|诺基亚|阿尔卡特|朗讯'
  )
GROUP BY p.country_code
ORDER BY applications DESC;


## `04_check_us_kind_codes.sql`

Checks whether filtering on kind code B dropped older US patents.


In [ ]:
-- Checks whether filtering on kind code B dropped older US patents.
SELECT
  SUBSTR(p.kind_code, 1, 1) AS kind,
  COUNT(DISTINCT p.publication_number) AS patents,
  MIN(p.grant_date) AS first_grant,
  MAX(p.grant_date) AS last_grant
FROM `patents-public-data.patents.publications` AS p,
  UNNEST(p.assignee_harmonized) AS a
WHERE p.country_code = 'US'
  AND p.grant_date > 0
  AND REGEXP_CONTAINS(
    UPPER(a.name),
    r'\bNOKIA\b|\bALCATEL\b|\bLUCENT TECHNOLOGIES\b|\bINFINERA\b'
  )
GROUP BY kind
ORDER BY patents DESC;


## `05_owner_names_all_years.sql`

Replaces queries 01 and 02. Kind A covers older US patents, B newer ones and E reissues.


In [ ]:
-- Replaces queries 01 and 02. Kind A covers older US patents, B newer ones and E reissues.
-- Design patents (S) and invention registrations (H) are left out.
-- The search is deliberately broad, and false matches get removed afterwards.
SELECT
  a.name AS owner,
  COUNT(DISTINCT p.publication_number) AS granted_patents,
  MIN(p.grant_date) AS first_grant,
  MAX(p.grant_date) AS last_grant
FROM `patents-public-data.patents.publications` AS p,
  UNNEST(p.assignee_harmonized) AS a
WHERE p.country_code = 'US'
  AND p.grant_date > 0
  AND SUBSTR(p.kind_code, 1, 1) IN ('A', 'B', 'E')
  AND REGEXP_CONTAINS(UPPER(a.name), r'NOKIA|ALCATEL|\bLUCENT\b|BELL LABS|INFINERA')
GROUP BY owner
ORDER BY granted_patents DESC;


## `06_create_portfolio_30y.sql`

Working table: filings since 1 Nov 1996, across all patent offices. Later queries use this instead of the 2.88 TB source.


In [ ]:
-- Working table: filings since 1 Nov 1996, across all patent offices. Later queries use this instead of the 2.88 TB source.
-- This is the exact text I ran. It used about 289 GB once. Sandbox tables expire after 60 days, so rerun it to rebuild the table.
CREATE SCHEMA IF NOT EXISTS nokia_analytics OPTIONS (location = 'US');
CREATE TEMP FUNCTION is_family(n STRING) AS ((REGEXP_CONTAINS(UPPER(n), r'NOKIA|ALCATEL|\bLUCENT\b|INFINERA') OR REGEXP_CONTAINS(n, r'诺基亚|阿尔卡特|朗讯|ノキア|노키아')) AND NOT REGEXP_CONTAINS(UPPER(n), r'NOKIAN|ENOKIAN|AUTOLIV') AND NOT REGEXP_CONTAINS(UPPER(n), r'MAILLEFER|SCANINTER') AND NOT REGEXP_CONTAINS(UPPER(n), r'LUCENT (MEDICAL|SKY|TRANS)') AND NOT REGEXP_CONTAINS(UPPER(n), r'LUCENT (ADVANCED|BIOSCIENCES)') AND NOT REGEXP_CONTAINS(UPPER(n), r'LUCENT OPTICS'));
CREATE OR REPLACE TABLE nokia_analytics.portfolio_30y AS SELECT p.publication_number, p.application_number, p.family_id, p.country_code, p.kind_code, p.priority_date, p.filing_date, p.publication_date, p.grant_date, ARRAY(SELECT DISTINCT a.name FROM UNNEST(p.assignee_harmonized) AS a WHERE is_family(a.name)) AS owners, ARRAY(SELECT i.name FROM UNNEST(p.inventor_harmonized) AS i) AS inventors, ARRAY(SELECT DISTINCT i.country_code FROM UNNEST(p.inventor_harmonized) AS i WHERE i.country_code != '') AS inventor_countries, ARRAY(SELECT c.code FROM UNNEST(p.cpc) AS c) AS cpc_codes, (SELECT t.text FROM UNNEST(p.title_localized) AS t WHERE t.language = 'en' LIMIT 1) AS title_en, (SELECT t.text FROM UNNEST(p.abstract_localized) AS t WHERE t.language = 'en' LIMIT 1) AS abstract_en, ARRAY(SELECT c.publication_number FROM UNNEST(p.citation) AS c WHERE c.publication_number != '') AS cited_publications FROM `patents-public-data.patents.publications` AS p WHERE p.filing_date >= 19961101 AND EXISTS (SELECT 1 FROM UNNEST(p.assignee_harmonized) AS a WHERE is_family(a.name));


## `07_check_portfolio_30y.sql`

Quick check of the working table: size, date range and coverage.


In [ ]:
-- Quick check of the working table: size, date range and coverage.
SELECT
  COUNT(*) AS publications,
  COUNT(DISTINCT application_number) AS applications,
  COUNT(DISTINCT family_id) AS families,
  COUNT(DISTINCT country_code) AS offices,
  MIN(filing_date) AS first_filing,
  MAX(filing_date) AS last_filing,
  COUNTIF(title_en IS NULL) AS without_english_title,
  COUNTIF(abstract_en IS NULL) AS without_english_abstract,
  COUNTIF(ARRAY_LENGTH(cpc_codes) = 0) AS without_cpc,
  COUNTIF(ARRAY_LENGTH(inventors) = 0) AS without_inventors,
  COUNTIF(ARRAY_LENGTH(inventor_countries) = 0) AS without_inventor_country,
  COUNTIF(ARRAY_LENGTH(cited_publications) = 0) AS without_citations
FROM nokia_analytics.portfolio_30y;


## `08_recount_us_30y.sql`

US recount from the working table: granted utility patents filed since 1 Nov 1996, by origin and top owner names.


In [ ]:
-- US recount from the working table: granted utility patents filed since 1 Nov 1996, by origin and top owner names.
-- Origin rows overlap because a publication can have multiple owners. Never sum origin rows to calculate a distinct-publication share.
-- The union row calculates the Alcatel/Lucent share at publication level; saved CSV snapshots require a rerun to include it.
WITH us AS (
    SELECT DISTINCT publication_number, o AS owner
    FROM nokia_analytics.portfolio_30y, UNNEST(owners) AS o
    WHERE country_code = 'US'
      AND grant_date > 0
      AND SUBSTR(kind_code, 1, 1) IN ('A', 'B', 'E')
), c AS (
    SELECT publication_number, owner,
        CASE
            WHEN REGEXP_CONTAINS(UPPER(owner), r'TCL|VACUUM|OPTRONICS|KABEL|CABLE|FIBRES|ESPACE|TELSPACE|MOBILE COMM|RADIOTELEPHONE|SUBMARCOM|SUBMARINE|TRANSP SOLUTION|ALSTHOM|ALSTHOLM|ALTHOM|ALTSHOM|COMMISSARIAT|INRIA|LUCENT KK|^TRT ') THEN 'Needs review'
            WHEN UPPER(owner) LIKE '%INFINERA%' THEN 'Infinera'
            WHEN REGEXP_CONTAINS(UPPER(owner), r'SHANGH[AI]{2} BELL') THEN 'Shanghai Bell'
            WHEN REGEXP_CONTAINS(UPPER(owner), r'S[IE]{2}M[EA]NS') THEN 'Nokia Siemens Networks'
            WHEN REGEXP_CONTAINS(UPPER(owner), r'^LUCENT|ATT CORP LUCENT') THEN 'Lucent'
            WHEN REGEXP_CONTAINS(UPPER(owner), r'ALCATEL|LUCENT') THEN 'Alcatel and Alcatel-Lucent'
            WHEN UPPER(owner) LIKE '%NOKIA%' THEN 'Nokia'
            ELSE 'Other'
        END AS origin
    FROM us
)
SELECT 'origin' AS row_type, origin AS label,
    COUNT(DISTINCT publication_number) AS patents,
    COUNT(DISTINCT owner) AS names,
    CAST(NULL AS FLOAT64) AS share_of_unique_publications
FROM c
GROUP BY origin
UNION ALL
SELECT 'total', 'Unique patents', COUNT(DISTINCT publication_number), COUNT(DISTINCT owner), NULL
FROM c
UNION ALL
SELECT 'total', 'Listings', COUNT(*), COUNT(DISTINCT owner), NULL
FROM c
UNION ALL
SELECT 'union', 'Alcatel and Lucent distinct union',
    COUNT(DISTINCT IF(origin IN ('Alcatel and Alcatel-Lucent', 'Lucent'), publication_number, NULL)),
    COUNT(DISTINCT IF(origin IN ('Alcatel and Alcatel-Lucent', 'Lucent'), owner, NULL)),
    COUNT(DISTINCT IF(origin IN ('Alcatel and Alcatel-Lucent', 'Lucent'), publication_number, NULL)) / COUNT(DISTINCT publication_number)
FROM c
UNION ALL (
    SELECT 'name', owner, COUNT(DISTINCT publication_number), 1, NULL
    FROM c
    GROUP BY owner
    ORDER BY 3 DESC
    LIMIT 15
)
ORDER BY row_type, patents DESC;


## `09_family_status.sql`

Sorts all families by grant evidence, using the working table only. Grant evidence is not current legal status:


In [ ]:
-- Sorts all families by grant evidence, using the working table only. Grant evidence is not current legal status:
-- rights may have lapsed or transferred, and grant_date is incomplete outside the US.
-- Saved CSV snapshots predate this office-aware heuristic and the clarified status labels.
CREATE OR REPLACE TABLE nokia_analytics.family_status AS
WITH pubs AS (
    SELECT family_id, country_code, kind_code, filing_date,
        EXISTS(
            SELECT 1 FROM UNNEST(owners) AS o
            WHERE NOT REGEXP_CONTAINS(UPPER(o), r'TCL|OPTRONICS|KABEL|CABLE|FIBRES|MOBILE COMM|RADIOTELEPHONE|SUBMARCOM|SUBMARINE')
        ) AS core_owner,
        CASE
            WHEN country_code = 'WO' THEN FALSE
            WHEN country_code = 'US' THEN grant_date > 0 AND REGEXP_CONTAINS(kind_code, r'^[ABE]')
            WHEN country_code = 'EP' THEN STARTS_WITH(kind_code, 'B')
            WHEN country_code = 'CN' THEN REGEXP_CONTAINS(kind_code, r'^[BC]')
            WHEN country_code = 'JP' THEN grant_date > 0
            WHEN NOT STARTS_WITH(kind_code, 'S') THEN grant_date > 0
            ELSE FALSE
        END AS grant_evidence
    FROM nokia_analytics.portfolio_30y
)
-- Legacy field names are retained for downstream compatibility; they represent grant evidence, not legal status.
SELECT family_id,
    LOGICAL_OR(core_owner) AS core_owner,
    LOGICAL_OR(grant_evidence) AS ever_granted,
    LOGICAL_OR(grant_evidence AND filing_date >= 20061101) AS granted_in_term,
    LOGICAL_OR(NOT grant_evidence AND country_code != 'WO' AND filing_date >= 20210101) AS recent_application,
    LOGICAL_OR(country_code = 'US' AND grant_evidence AND filing_date >= 20061101) AS us_in_term,
    COUNT(DISTINCT IF(grant_evidence AND filing_date >= 20061101, country_code, NULL)) AS offices_in_term,
    MIN(filing_date) AS first_filing,
    MAX(filing_date) AS last_filing
FROM pubs
GROUP BY family_id;

SELECT CASE
    WHEN NOT core_owner THEN '5 Sold-business names only'
    WHEN granted_in_term THEN '1 Grant evidence, within 20 years'
    WHEN recent_application THEN '2 Recent application, no grant detected'
    WHEN ever_granted THEN '3 Grant evidence, past 20 years'
    ELSE '4 No grant detected, older filing'
END AS status,
    COUNT(*) AS families,
    COUNTIF(us_in_term) AS with_us_in_term,
    ROUND(AVG(offices_in_term), 2) AS avg_offices_in_term
FROM nokia_analytics.family_status
GROUP BY status
ORDER BY status;


## `10_in_term_by_office.sql`

Families with a detected grant within 20 years, by patent office. This is the exact text I ran.


In [ ]:
-- Families with a detected grant within 20 years, by patent office. This is the exact text I ran.
SELECT p.country_code, COUNT(DISTINCT p.family_id) AS families FROM nokia_analytics.portfolio_30y AS p JOIN nokia_analytics.family_status AS s USING (family_id) WHERE s.core_owner AND s.granted_in_term AND p.filing_date >= 20061101 AND p.country_code != 'WO' AND NOT STARTS_WITH(p.kind_code, 'S') AND (p.grant_date > 0 OR REGEXP_CONTAINS(p.kind_code, r'^[BCE]')) GROUP BY p.country_code ORDER BY families DESC LIMIT 15;


## `11_new_families_by_year.sql`

New families per year, using the first filing date in the working table. This is part of the exact query I ran.


In [ ]:
-- New families per year, using the first filing date in the working table. This is part of the exact query I ran.
SELECT DIV(first_filing, 10000) AS yr, COUNT(*) AS new_families FROM nokia_analytics.family_status GROUP BY yr ORDER BY yr;


## `12_office_filings_by_family.sql`

Filings in the four focus offices for every family in the table, found by family rather than owner name, which recovers filings with a missing owner name. This is the exact text I ran, at about 10 GB.


In [ ]:
-- Filings in the four focus offices for every family in the table, found by family rather than owner name, which recovers filings with a missing owner name. This is the exact text I ran, at about 10 GB.
SELECT DIV(p.filing_date, 10000) AS yr, COUNT(DISTINCT IF(p.country_code = 'US', p.application_number, NULL)) AS us, COUNT(DISTINCT IF(p.country_code = 'EP', p.application_number, NULL)) AS ep, COUNT(DISTINCT IF(p.country_code = 'CN', p.application_number, NULL)) AS cn, COUNT(DISTINCT IF(p.country_code = 'JP', p.application_number, NULL)) AS jp FROM `patents-public-data.patents.publications` AS p WHERE p.country_code IN ('US', 'EP', 'CN', 'JP') AND p.filing_date >= 19961101 AND p.family_id IN (SELECT family_id FROM nokia_analytics.family_status) GROUP BY yr ORDER BY yr;


## `13_spot_check_sample.sql`

Repeatable random sample for the register spot-check: up to 20 US, 10 EP, 5 CN and 5 JP granted documents in core families. This is the exact text I ran.


In [ ]:
-- Repeatable random sample for the register spot-check: up to 20 US, 10 EP, 5 CN and 5 JP granted documents in core families. This is the exact text I ran.
WITH p AS (SELECT t.publication_number, t.country_code, t.kind_code, t.filing_date, t.grant_date, t.owners[SAFE_OFFSET(0)] AS owner, ROW_NUMBER() OVER (PARTITION BY t.country_code ORDER BY FARM_FINGERPRINT(CONCAT('spot-check-2026', t.publication_number))) AS rn FROM nokia_analytics.portfolio_30y AS t JOIN nokia_analytics.family_status AS s USING (family_id) WHERE s.core_owner AND s.granted_in_term AND t.country_code IN ('US', 'EP', 'CN', 'JP') AND t.filing_date >= 20061101 AND NOT STARTS_WITH(t.kind_code, 'S') AND (t.grant_date > 0 OR REGEXP_CONTAINS(t.kind_code, r'^[BCE]'))) SELECT publication_number, country_code, kind_code, filing_date, grant_date, owner FROM p WHERE (country_code = 'US' AND rn <= 20) OR (country_code = 'EP' AND rn <= 10) OR (country_code IN ('CN', 'JP') AND rn <= 5) ORDER BY country_code, publication_number;


## `14_mean_median_patent_measures.sql`

Mean, median and spread for time to grant and filings per core family. This is the exact text I ran. Offices include WO, so they overstate national coverage.


In [ ]:
-- Mean, median and spread for time to grant and filings per core family. This is the exact text I ran. Offices include WO, so they overstate national coverage.
WITH g AS (SELECT DISTINCT t.application_number, t.country_code, t.family_id, DATE_DIFF(PARSE_DATE('%Y%m%d', CAST(t.grant_date AS STRING)), PARSE_DATE('%Y%m%d', CAST(t.filing_date AS STRING)), DAY) / 365.25 AS years FROM nokia_analytics.portfolio_30y AS t JOIN nokia_analytics.family_status AS s USING (family_id) WHERE s.core_owner AND s.granted_in_term AND t.filing_date BETWEEN 20061101 AND 20151231 AND t.grant_date > 0 AND t.country_code IN ('US', 'EP', 'CN') AND REGEXP_CONTAINS(t.kind_code, r'^[BC]')), f AS (SELECT family_id, COUNT(DISTINCT application_number) AS apps, COUNT(DISTINCT country_code) AS offices FROM nokia_analytics.portfolio_30y AS t JOIN nokia_analytics.family_status AS s USING (family_id) WHERE s.core_owner AND s.granted_in_term GROUP BY family_id) SELECT CONCAT(country_code, ' years to grant') AS measure, COUNT(*) AS n, ROUND(AVG(years), 2) AS mean, ROUND(APPROX_QUANTILES(years, 100)[OFFSET(50)], 2) AS median, ROUND(APPROX_QUANTILES(years, 100)[OFFSET(10)], 2) AS p10, ROUND(APPROX_QUANTILES(years, 100)[OFFSET(90)], 2) AS p90 FROM g GROUP BY country_code UNION ALL SELECT 'Applications per core family', COUNT(*), ROUND(AVG(apps), 2), APPROX_QUANTILES(apps, 100)[OFFSET(50)], APPROX_QUANTILES(apps, 100)[OFFSET(10)], APPROX_QUANTILES(apps, 100)[OFFSET(90)] FROM f UNION ALL SELECT 'Offices per core family', COUNT(*), ROUND(AVG(offices), 2), APPROX_QUANTILES(offices, 100)[OFFSET(50)], APPROX_QUANTILES(offices, 100)[OFFSET(10)], APPROX_QUANTILES(offices, 100)[OFFSET(90)] FROM f ORDER BY measure;


## `15_rivals_30y.sql`

Query preserved from the corresponding SQL file.


In [ ]:
CREATE OR REPLACE TABLE nokia_analytics.rivals_30y AS SELECT p.publication_number, p.application_number, p.family_id, p.country_code, p.kind_code, p.priority_date, p.filing_date, p.grant_date, ARRAY(SELECT DISTINCT a.name FROM UNNEST(p.assignee_harmonized) AS a WHERE REGEXP_CONTAINS(UPPER(a.name), r'\bCIENA\b|\bCISCO\b|ACACIA COMM|\bLUXTERA\b|ARISTA NETWORKS|\bHUAWEI\b') OR REGEXP_CONTAINS(a.name, r'华为')) AS owners, ARRAY(SELECT c.code FROM UNNEST(p.cpc) AS c) AS cpc_codes, (SELECT t.text FROM UNNEST(p.title_localized) AS t WHERE t.language = 'en' LIMIT 1) AS title_en FROM `patents-public-data.patents.publications` AS p WHERE p.filing_date >= 19961101 AND EXISTS (SELECT 1 FROM UNNEST(p.assignee_harmonized) AS a WHERE REGEXP_CONTAINS(UPPER(a.name), r'\bCIENA\b|\bCISCO\b|ACACIA COMM|\bLUXTERA\b|ARISTA NETWORKS|\bHUAWEI\b') OR REGEXP_CONTAINS(a.name, r'华为'));


## `16_codebook_sample.sql`

Sampling limits exceed the final company slot quotas to provide spares for replacements.


In [ ]:
-- Sampling limits exceed the final company slot quotas to provide spares for replacements.
-- Saved sample CSVs predate the explicit Huawei match and Other fallback; this edit does not regenerate them.
WITH b AS (
    SELECT 'Nokia' AS co, family_id, publication_number, cpc_codes
    FROM nokia_analytics.portfolio_30y
    WHERE country_code = 'US' AND grant_date > 0 AND filing_date >= 20061101 AND STARTS_WITH(kind_code, 'B')
    UNION ALL
    SELECT CASE
        WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'CIENA') THEN 'Ciena'
        WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'CISCO|ACACIA|LUXTERA') THEN 'Cisco'
        WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'ARISTA') THEN 'Arista'
        WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'HUAWEI') THEN 'Huawei'
        ELSE 'Other'
    END, family_id, publication_number, cpc_codes
    FROM nokia_analytics.rivals_30y
    WHERE country_code = 'US' AND grant_date > 0 AND filing_date >= 20061101 AND STARTS_WITH(kind_code, 'B')
), p AS (
    SELECT co, family_id, publication_number,
        CASE WHEN EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c WHERE REGEXP_CONTAINS(c, r'^(H04B10/|H04J14/|H04Q11/00)')) THEN 1 ELSE 0 END AS opt,
        CASE WHEN EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c WHERE STARTS_WITH(c, 'H04L45/')) THEN 1 ELSE 0 END AS ipr,
        CASE WHEN EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c WHERE REGEXP_CONTAINS(c, r'^(H04L49/|H04L47/)')) THEN 1 ELSE 0 END AS dcs
    FROM b
), l AS (
    SELECT 'OPT' AS pool, co, family_id, MIN(publication_number) AS pub FROM p WHERE opt = 1 GROUP BY co, family_id
    UNION ALL
    SELECT 'IPR', co, family_id, MIN(publication_number) FROM p WHERE ipr = 1 GROUP BY co, family_id
    UNION ALL
    SELECT 'DCS', co, family_id, MIN(publication_number) FROM p WHERE dcs = 1 GROUP BY co, family_id
), r AS (
    SELECT *, COUNT(*) OVER (PARTITION BY pool, co) AS pool_size,
        ROW_NUMBER() OVER (PARTITION BY pool, co ORDER BY FARM_FINGERPRINT(CONCAT('codebook-2026', family_id))) AS rn
    FROM l
)
SELECT pool, co, ANY_VALUE(pool_size) AS pool_size, STRING_AGG(pub, ' ' ORDER BY rn) AS sample
FROM r
WHERE rn <= CASE co WHEN 'Nokia' THEN 56 WHEN 'Huawei' THEN 17 WHEN 'Arista' THEN 17 ELSE 30 END
GROUP BY pool, co
ORDER BY pool, co;


## `17_codebook_sample_h04l49.sql`

Query preserved from the corresponding SQL file.


In [ ]:
WITH b AS (SELECT 'Nokia' AS co, family_id, publication_number, cpc_codes FROM nokia_analytics.portfolio_30y WHERE country_code = 'US' AND grant_date > 0 AND filing_date >= 20061101 AND STARTS_WITH(kind_code, 'B') UNION ALL SELECT CASE WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'CIENA') THEN 'Ciena' WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'CISCO|ACACIA|LUXTERA') THEN 'Cisco' WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'ARISTA') THEN 'Arista' ELSE 'Huawei' END, family_id, publication_number, cpc_codes FROM nokia_analytics.rivals_30y WHERE country_code = 'US' AND grant_date > 0 AND filing_date >= 20061101 AND STARTS_WITH(kind_code, 'B')), l AS (SELECT co, family_id, MIN(publication_number) AS pub FROM b WHERE EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c WHERE STARTS_WITH(c, 'H04L49/')) GROUP BY co, family_id), r AS (SELECT *, COUNT(*) OVER (PARTITION BY co) AS pool_size, ROW_NUMBER() OVER (PARTITION BY co ORDER BY FARM_FINGERPRINT(CONCAT('codebook-2026', family_id))) AS rn FROM l) SELECT co, ANY_VALUE(pool_size) AS pool_size, STRING_AGG(pub, ' ' ORDER BY rn) AS sample FROM r WHERE rn <= CASE co WHEN 'Nokia' THEN 60 ELSE 30 END GROUP BY co ORDER BY co;


## `18_validation_sample_metadata.sql`

Family ID and priority date for the 421 labelled patents, for data/validation/sample_metadata.csv.


In [ ]:
-- Family ID and priority date for the 421 labelled patents, for data/validation/sample_metadata.csv.
-- Run on 1 November 2026 with the full list of publication numbers from data/validation/validation_slots.csv in the IN clause.
-- It returned one concatenated cell, which I split into the CSV. Applicant and current assignee in that CSV come from the saved Google Patents pages.
SELECT STRING_AGG(CONCAT(publication_number,'|',CAST(family_id AS STRING),'|',CAST(priority_date AS STRING),'|',CAST(filing_date AS STRING),'|',ARRAY_TO_STRING(owners,';')),'#') AS x FROM (SELECT publication_number,family_id,priority_date,filing_date,owners FROM nokia_analytics.portfolio_30y UNION ALL SELECT publication_number,family_id,priority_date,filing_date,owners FROM nokia_analytics.rivals_30y) WHERE publication_number IN (/* the 421 publication numbers */);


## Saved outputs and validation

Saved query outputs live under `data/raw/`, `data/processed/`, and `data/validation/`. Run `python3 src/check_outputs.py` after refreshing outputs. The publication-level Alcatel/Lucent union added to `sql/08_recount_us_30y.sql` still requires a BigQuery rerun before any percentage can be reported.
